# Assignment 6: Attention (please!)

---

## Task 2) Sentiment Analysis

In this task, we'll use the kaggle Rotten Tomatoes Dataset for this exercise: [Source and Download instructions](https://www.kaggle.com/c/sentiment-analysis-on-movie-reviews/data).
The dataset is comprised of tab-separated files with phrases from the Rotten Tomatoes dataset.
The train/test split has been preserved for the purposes of benchmarking, but the sentences have been shuffled from their original order.
Each sentence has been parsed into many phrases (chunks) using the Stanford parser.
Each phrase has a `PhraseId`, each sentence a `SentenceId`.
Phrases that are repeated (such as short/common words) are only included once in the data.

### Data

Rotten Tomatoes Dataset: `train.tsv` contains the phrases and their associated sentiment labels.
We have additionally provided a `SentenceId` so that you can track which phrases belong to a single sentence.
`test.tsv` contains just phrases; use your model to assign a sentiment label to each phrase.

The sentiment labels are:

* 0 - negative
* 1 - somewhat negative
* 2 - neutral
* 3 - somewhat positive
* 4 - positive

### GloVe Word Embeddings

Use GloVe word embeddings for your `nn.Embedding` layer, there is a number of pretrained models for English available in the `torchtext` module.
You are free to  use any kind of attention and architecture you like.
Just remember that the basic form for attention based networks is always and encoder / Decoder architecture.
Use `torchtext.vocab.GloVe` to get started quickly with the word embeddings.


*In this Jupyter Notebook, we will provide the steps to solve this task and give hints via functions & comments. However, code modifications (e.g., function naming, arguments) and implementation of additional helper functions & classes are allowed. The code aims to help you get started.*

---

In [1]:
# Dependencies
import os
import re
import tqdm
import string
import numpy as np
import pandas as pd
import sklearn.metrics as sklearn_metrics
from sklearn.model_selection import StratifiedKFold

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

### Prepare the Data

1.1 As always: conduct some data preprocessing.

1.2 Download and prepare the GloVe word embeddings. You'll need it for the modeling part such as nn.Embedding.

1.3 Create a PyTorch Dataset class which handles your tokenized data with respect to input and (class) labels.

In [2]:
def load_sentiment_dataset(filepath):
    """Loads all phrase instances and returns them as a dataframe."""
    ### YOUR CODE HERE
    
    return pd.read_csv(filepath, header=0, sep='\t')
    
    ### END YOUR CODE

In [3]:
def preprocess(dataframe):
    """Preprocesses and tokenizes the given theses titles for further use."""
    ### YOUR CODE HERE
    
    def _preprocss_fn(text):
        remove_pun = str.maketrans(string.punctuation, ' '*len(string.punctuation))
        remove_digits = str.maketrans(string.digits, ' '*len(string.digits))
        text = text.translate(remove_digits)
        text = text.translate(remove_pun)
        text = re.sub(' {2,}', ' ', text)
        return text.lower()
    
    dataframe = dataframe.copy()
    
    # Remove punctuation, digits and lowercase phrases
    dataframe["Phrase"] = dataframe["Phrase"].apply(lambda s: _preprocss_fn(s))

    # Filter out empty phrases
    dataframe = dataframe[dataframe["Phrase"].str.len() > 1]

    # Reset index of dataframe
    dataframe = dataframe.reset_index(drop=True)

    # Simple tokenization of phrases
    dataframe["tokenized"] = [title.split() for title in dataframe["Phrase"].values]

    return dataframe

    ### END YOUR CODE

In [4]:
# Load and preprocess dataset
train_dataframe = load_sentiment_dataset("data/rotten_tomatoes_train.tsv")
train_dataframe = preprocess(train_dataframe)

# Map for formatting labels
IDX2SENTIMENT = {0: "negative", 1: "somewhat negative", 2: "neutral",
                 3: "somewhat positive", 4: "positive"}

# Test labels not available and submission to Kaggle required
# test_dataframe = load_sentiment_dataset("data/rotten_tomatoes_test.tsv")
# test_dataframe = preprocess(test_dataframe)

print(f"Num train dataset: {len(train_dataframe)}")
# print(f"Num test dataset: {len(test_dataframe)}")

Num train dataset: 155880


In [5]:
### Download the pre-trained (english) GloVe embeddings
from torchtext.vocab import GloVe

# Prepare glove embeddings
UNK_TOKEN = "<unk>"
PAD_TOKEN = "<pad>"

def append_special(glove, special, vec=None):
    glove.itos.append(special)
    glove.stoi[special] = glove.itos.index(special)
    if vec is None:
        vec = torch.zeros(1, glove.vectors.size(1))
    glove.vectors = torch.cat((glove.vectors, vec))
    return glove

glove = GloVe(name="6B", dim=50)

# We need to add some special tokens
glove = append_special(glove, UNK_TOKEN)
glove = append_special(glove, PAD_TOKEN)

/Users/seebergerph/anaconda3/envs/seqlrn/lib/python3.10/site-packages/torchtext/vocab/__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
/Users/seebergerph/anaconda3/envs/seqlrn/lib/python3.10/site-packages/torchtext/utils.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)


In [6]:
class RottenTomatoesDataset(Dataset):
    def __init__(self, dataset, labels, glove, unk="<unk>"):
        self.data, self.labels = [], []
        for tokens, label in zip(dataset, labels):
            # Create inputs; map tokens to ids
            self.data.append(torch.stack([
                torch.tensor(glove.stoi.get(w, glove.stoi.get(unk)), dtype=torch.long) for w in tokens
            ]))

            # Create labels; already an integer
            self.labels.append(label)


    def __len__(self):
        return len(self.data)


    def __getitem__(self, idx):
        # Returns one input and label sample
        return self.data[idx], self.labels[idx]

### Train and Evaluate

2.1 Implement and reuse your RNN-based classifciation models for the sentiment classification task. 

2.2 Train and evaluate your models by performing a train-test-split on the `train.tsv` file.

2.3 Check and compare your classification results with some publicly available baselines (there are plenty of on the internet).

2.4 Visualize the attention weights for the words and pick some nice samples for each sentiment category!

In [7]:
### TODO: 2.1 Implement RNN classifier (nn.Module)
### Notice: Think about padding for batch sizes > 1
### Notice: 'torch.nn.utils.rnn' provides functionality

### YOUR CODE HERE

from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence, pad_sequence

class GRU_Classifier(nn.Module):
    def __init__(self, glove, hidden_dim, num_classes, with_attention=False):
        super(GRU_Classifier, self).__init__()
        self.with_attention = with_attention

        self.embedding = nn.Embedding.from_pretrained(glove.vectors, freeze=True)

        self.rnn = nn.GRU(
            input_size=glove.dim,
            hidden_size=hidden_dim,
            bidirectional=False,
            num_layers=1
        )

        if self.with_attention:
            self.attention = DotProductAttention(
                q_dim=hidden_dim,
                k_dim=hidden_dim,
                v_dim=hidden_dim,
                batch_first=False
            )

        self.fc = nn.Linear(hidden_dim, num_classes)

    
    def forward(self, X, lengths, hidden=None):
        embeddings = self.embedding(X)

        # Packed squence helps avoid unneccsary computation
        padded_seq = pack_padded_sequence(embeddings, lengths)

        outputs, hidden_states = self.rnn(padded_seq, hidden)

        # If tuple (h_n, c_n) containts cell state c_n then select h_n
        if isinstance(hidden_states, tuple):
            hidden = hidden_states[0]
        else:
            hidden = hidden_states

        if self.with_attention:
            # Padding for packed sequences
            padded_seq , lens = pad_packed_sequence(outputs)

            # Get top hidden states and add seq_len 1 for attention
            hidden = hidden[-1].unsqueeze(0)

            # Apply dot product attention
            clf_input, weights = self.attention(hidden, padded_seq, padded_seq)
        else:
            clf_input, weights = hidden, None

        # Apply classifier with hidden states
        logits = self.fc(clf_input.squeeze(0))

        return logits, hidden_states, weights
    

class DotProductAttention(nn.Module):
    def __init__(self, q_dim, k_dim, v_dim, batch_first=False):
        super().__init__()
        self.batch_first = batch_first
        self.scale = np.sqrt(k_dim)
        self.softm = nn.Softmax(dim=2)

        self.W_q = nn.Linear(q_dim, k_dim)
        self.W_k = nn.Linear(q_dim, k_dim)
        self.W_v = nn.Linear(k_dim, v_dim)


    def forward(self, x_q, x_k, x_v):
        Q = self.W_q(x_q)
        K = self.W_k(x_k)
        V = self.W_v(x_v)
        A, attn_weights = self.scaled_dp_attention(Q, K, V)
        return A, attn_weights
    

    def scaled_dp_attention(self, q, k, v):
        if not self.batch_first:
            k = k.permute(1, 0, 2)
            q = q.permute(1, 0, 2)
            v = v.permute(1, 0, 2)
        attn_weights = torch.matmul(q, k.transpose(1, 2))
        attn_weights = attn_weights / self.scale
        attn_weights = self.softm(attn_weights)
        A = torch.matmul(attn_weights, v)

        if not self.batch_first:
            A = A.permute(1, 0, 2)
            attn_weights = attn_weights.permute(1, 0, 2)
        return A, attn_weights


class SequencePadder():
    def __init__(self, symbol) -> None:
        self.symbol = symbol

    def __call__(self, batch):
        sorted_batch = sorted(batch, key=lambda x: x[0].size(0), reverse=True)
        sequences = [x[0] for x in sorted_batch]
        labels = [x[1] for x in sorted_batch]
        padded = pad_sequence(sequences, padding_value=self.symbol)
        lengths = torch.LongTensor([len(x) for x in sequences])
        return padded, torch.LongTensor(labels), lengths


### END YOUR CODE

In [8]:
### TODO: 2.2 Implement the train functionality

### YOUR CODE HERE

def train(model, dataloader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0

    optimizer.zero_grad()

    predictions = []
    ground_truth = []
    for inputs, labels, lengths in tqdm.tqdm(dataloader, desc="Train"):
        inputs = inputs.to(device)
        labels = labels.to(device)

        logits, hidden, weights = model(inputs, lengths)

        preds = torch.argmax(logits, axis=-1)

        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        running_loss += loss.item()
        predictions.extend(preds.numpy().tolist())
        ground_truth.extend(labels.numpy().tolist())

    running_loss = running_loss / len(dataloader)
    return predictions, ground_truth, running_loss

### END YOUR CODE

In [9]:
### TODO: 2.2 Implement the evaluation functionality

### YOUR CODE HERE

def eval(model, dataloader, criterion, device, return_attn_dict=False):
    model.eval()

    running_loss = 0.0

    predictions = []
    ground_truth = []
    sentences = []
    attn_weights = []
    with torch.no_grad():
        for inputs, labels, lengths in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            logits, hidden, weights = model(inputs, lengths)

            preds = torch.argmax(logits, axis=-1)

            loss = criterion(logits, labels)

            running_loss += loss.item()
            predictions.extend(preds.cpu().numpy().tolist())
            ground_truth.extend(labels.cpu().numpy().tolist())

            if return_attn_dict:
                sentences.append(inputs.cpu().numpy())
                attn_weights.append(weights.cpu().squeeze().numpy())

    running_loss = running_loss / len(dataloader)

    if return_attn_dict:
        attn_dict = {"tokens": sentences, "weights": attn_weights}
        return predictions, ground_truth, attn_dict
    else:
        return predictions, ground_truth, running_loss


def compute_metrics(preds, labels):
    return {
        'f1': sklearn_metrics.f1_score(y_true=labels, y_pred=preds, average="macro"),
        'prec': sklearn_metrics.precision_score(y_true=labels, y_pred=preds, average="macro"),
        'recall': sklearn_metrics.recall_score(y_true=labels, y_pred=preds, average="macro"),
        'acc': sklearn_metrics.accuracy_score(y_true=labels, y_pred=preds)
    }

### END YOUR CODE

In [10]:
### TODO: 2.3 Initialize and train the RNN Language Model for X epochs + Evaluation

# Training parameters
SEED = 42
EPOCHS = 10
BATCH_SIZE = 16

LEARNING_RATE = 0.0001

DEVICE = "cpu" # 'cpu', 'mps' or 'cuda'
LABEL_COL = "Sentiment"
PAD_IDX = glove.stoi[PAD_TOKEN]

# Model parameters
VARIANT = "gru"
EMBEDDING_DIM = 256
HIDDEN_DIM = 256

### YOUR CODE HERE


folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(
    train_dataframe,
    train_dataframe[LABEL_COL].values
)

VARIANTS = [("gru", False), ("gru", True)]

# Iterate variatants
results_all = []
models_all = {}
for rnn_variant, with_attention in VARIANTS:
    model_name = rnn_variant
    if with_attention:
        model_name += "-attention"

    # Iterate folds
    for fold, (train_idx, val_idx) in enumerate(folds, start=1):
        train_data = train_dataframe.iloc[train_idx]
        val_data = train_dataframe.iloc[val_idx]

        # Prepare samples
        train_labels = train_data[LABEL_COL].values
        val_labels = val_data[LABEL_COL].values
        train_data = train_data.tokenized
        val_data = val_data.tokenized

        # Use batch_size=1 to avoid padding
        train_dataset = RottenTomatoesDataset(train_data, train_labels, glove=glove, unk=UNK_TOKEN)
        train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, collate_fn=SequencePadder(PAD_IDX))

        # Use batch_size=1 to avoid padding
        val_dataset = RottenTomatoesDataset(val_data, val_labels, glove=glove, unk=UNK_TOKEN)
        val_dataloader = DataLoader(val_dataset, batch_size=1, collate_fn=SequencePadder(PAD_IDX))

        model = GRU_Classifier(
            glove=glove,
            hidden_dim=HIDDEN_DIM,
            num_classes=len(IDX2SENTIMENT),
            with_attention=with_attention
        )
        model = model.to(DEVICE)

        criterion = nn.CrossEntropyLoss(reduction="mean")

        optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

        best_epoch = -1
        best_f1 = 0
        for epoch in range(1, EPOCHS + 1):
            print(f"Epoch {epoch} of {EPOCHS}")
            print("-" * 20)

            # Training step
            train_preds, train_labels, train_loss = train(
                model=model,
                dataloader=train_dataloader,
                criterion=criterion,
                optimizer=optimizer,
                device=DEVICE
            )

            # Evaluation step
            val_preds, val_labels, val_loss = eval(
                model=model,
                dataloader=val_dataloader,
                criterion=criterion,
                device=DEVICE
            )

            # Compute all metrics
            train_metrics = compute_metrics(preds=train_preds, labels=train_labels)
            val_metrics = compute_metrics(preds=val_preds, labels=val_labels)

            print(f"Train loss: {train_loss:.4f} | Train F1: {train_metrics['f1']:.4f}")
            print(f"Val loss: {val_loss:.4f} | Val F1: {val_metrics['f1']:.4f}")

            # Save best model by evaluation loss
            if best_f1 <= val_metrics["f1"]:
                best_epoch = epoch
                best_f1 = val_metrics["f1"]
                print(f"Saving best model ...")
                torch.save(model.state_dict(), f"data/best_{model_name}_clf.pt")

        # Load best and final model
        print(f"Best epoch: {best_epoch}")
        print(f"Best F1: {best_f1:.4f}")
        print(f"Loading best model ...")
        model.load_state_dict(torch.load(f"data/best_{model_name}_clf.pt"))

        # Compute val metrics
        val_preds, val_labels, val_loss = eval(
            model=model, dataloader=val_dataloader, criterion=criterion, device=DEVICE
        )
        val_metrics = compute_metrics(preds=val_preds, labels=val_labels)
        results_all.append({"variant": model_name, **val_metrics})
        print(val_metrics)

        models_all[model_name] = model

        # Remove for cross-validation results
        break

### END YOUR CODE

Epoch 1 of 10
--------------------


Train: 100%|██████████| 7794/7794 [00:59<00:00, 130.61it/s]


Train loss: 1.1128 | Train F1: 0.3022
Val loss: 1.0262 | Val F1: 0.3310
Saving best model ...
Epoch 2 of 10
--------------------


Train: 100%|██████████| 7794/7794 [00:58<00:00, 132.15it/s]


Train loss: 1.0144 | Train F1: 0.3620
Val loss: 1.0007 | Val F1: 0.3523
Saving best model ...
Epoch 3 of 10
--------------------


...

Train loss: 0.9256 | Train F1: 0.4597
Val loss: 0.9350 | Val F1: 0.4574
Saving best model ...
Epoch 9 of 10
--------------------


Train: 100%|██████████| 7794/7794 [01:21<00:00, 95.18it/s] 


Train loss: 0.9141 | Train F1: 0.4697
Val loss: 0.9286 | Val F1: 0.4642
Saving best model ...
Epoch 10 of 10
--------------------


Train: 100%|██████████| 7794/7794 [01:18<00:00, 99.66it/s] 


Train loss: 0.9023 | Train F1: 0.4811
Val loss: 0.9228 | Val F1: 0.4739
Saving best model ...
Best epoch: 10
Best F1: 0.4739
Loading best model ...
{'f1': 0.4739326745986155, 'prec': 0.563824217868717, 'recall': 0.4442294966863744, 'acc': 0.6195150115473441}


In [11]:
results_df = pd.DataFrame(results_all)
results_df = results_df.groupby("variant").mean()
results_df

,f1,prec,recall,acc
variant,,,,
gru,0.449227,0.560656,0.417004,0.613902
gru-attention,0.473933,0.563824,0.444229,0.619515


In [19]:
### TODO: 2.4 Visualize the attention weights

### YOUR CODE HERE

import random
import warnings
import collections
import matplotlib
from IPython.display import display, HTML

warnings.filterwarnings("ignore")

def colorize_sentence(words, color_array, cmap = matplotlib.cm.get_cmap('RdBu')):
    assert(len(words) == len(color_array))
    # color_array is an array of numbers between 0 and 1 of length equal to words
    template = '<span class="barcode"; style="color: black; background-color: {}">{}</span>'
    colored_string = ''
    for word, color in zip(words, color_array):
        color = matplotlib.colors.rgb2hex(cmap(color.item())[:3])
        colored_string += template.format(color, '&nbsp' + word + '&nbsp')
    return colored_string


# Get predictions and attention weights
val_dataset = RottenTomatoesDataset(val_data, val_labels, glove=glove, unk=UNK_TOKEN)
val_dataloader = DataLoader(val_dataset, batch_size=1, collate_fn=SequencePadder(PAD_IDX))
_preds, labels, attn_dict = eval(
    model=models_all["gru-attention"], dataloader=val_dataloader,  device=DEVICE,
    criterion=nn.CrossEntropyLoss(reduction="mean"), return_attn_dict=True
)


# Make word color maps
attn_weights = collections.defaultdict(list)
colorized_sentences = collections.defaultdict(list)
for tokens, weights, pred in zip(attn_dict["tokens"], attn_dict["weights"], _preds):
    words = [glove.itos[token[0]] for token in tokens]
    weights = np.array(weights) / np.max(weights)
    if len(words) < 5: continue
    attn_weights[IDX2SENTIMENT[pred]].append(weights)
    colorized_sentences[IDX2SENTIMENT[pred]].append(colorize_sentence(words, weights))

# Show samples for sentiments
for sentiment in IDX2SENTIMENT.values():
    print(f"Sentiment: {sentiment}")
    senti_attn_weights = attn_weights[sentiment]
    senti_color_sents = colorized_sentences[sentiment]

    # Display 5 random sentences
    num_samples = 5
    for i in [random.randint(0, len(senti_color_sents)) for _ in range(num_samples)]:
        display(HTML(senti_color_sents[i]))

### END YOUR CODE

Sentiment: negative


Sentiment: somewhat negative


Sentiment: neutral


Sentiment: somewhat positive


Sentiment: positive
